## Imports

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import warnings
import time
import gc

## Constants

In [25]:
DATA_FOLDER = '/net/tscratch/people/tutorial237/EnsembleAi2026/task3/data'
DATA_PATH    = f"{DATA_FOLDER}/data.csv"
DEVICES_PATH = f"{DATA_FOLDER}/devices.csv"
SAMPLE_FRAC  = 0.05   # e.g. 0.1 to use 10%
# ───────────────────────────────────────────────────────────────────────────────

TEMP_COLS = [f"t{i}" for i in range(1, 14)]
TARGET    = "x2"
SPLIT_DATE_TRAIN_END = "2025-04-30"
SPLIT_DATE_VAL_END   = "2025-06-30"

## Load

In [16]:
TEMP_COLS = [f"t{i}" for i in range(1, 14)]
DTYPES    = {c: "float32" for c in TEMP_COLS + ["x1", "x2", "x3"]}
DTYPES.update({"deviceId": "str", "deviceType": "int8"})

In [21]:
def aggregate_chunk(chunk: pd.DataFrame) -> pd.DataFrame:
    chunk["year"]  = chunk["timedate"].str[:4].astype("int16")
    chunk["month"] = chunk["timedate"].str[5:7].astype("int8")
 
    return chunk.groupby(["deviceId", "year", "month"]).agg(
        # target
        x2_mean  = ("x2", "mean"),
        x2_std   = ("x2", "std"),
        x2_count = ("x2", "count"),
 
        # all temperature sensors
        t1_mean  = ("t1",  "mean"),  t1_std  = ("t1",  "std"),
        t2_mean  = ("t2",  "mean"),  t2_std  = ("t2",  "std"),
        t3_mean  = ("t3",  "mean"),  t3_std  = ("t3",  "std"),
        t4_mean  = ("t4",  "mean"),  t4_std  = ("t4",  "std"),
        t5_mean  = ("t5",  "mean"),  t5_std  = ("t5",  "std"),
        t6_mean  = ("t6",  "mean"),  t6_std  = ("t6",  "std"),
        t7_mean  = ("t7",  "mean"),  t7_std  = ("t7",  "std"),
        t8_mean  = ("t8",  "mean"),  t8_std  = ("t8",  "std"),
        t9_mean  = ("t9",  "mean"),  t9_std  = ("t9",  "std"),
        t10_mean = ("t10", "mean"),  t10_std = ("t10", "std"),
        t11_mean = ("t11", "mean"),  t11_std = ("t11", "std"),
        t12_mean = ("t12", "mean"),  t12_std = ("t12", "std"),
        t13_mean = ("t13", "mean"),  t13_std = ("t13", "std"),
 
        # operating features
        x1_mean = ("x1", "mean"),  x1_std = ("x1", "std"),
        x3_mode = ("x3", lambda s: s.mode().iloc[0] if len(s) else None),
 
        # device type (constant per device, take first)
        deviceType = ("deviceType", "first"),
    ).reset_index()

In [18]:
def combine_partial_aggs(chunks: list[pd.DataFrame]) -> pd.DataFrame:
    """
    Concatenate partial chunk aggregations and re-aggregate.
    Simple mean-of-means is fine here because chunks are same size (~500k rows),
    but re-grouping is exact regardless.
    """
    raw = pd.concat(chunks, ignore_index=True)
    return raw.groupby(["deviceId", "year", "month"]).agg(
        x2_mean  = ("x2_mean",  "mean"),
        x2_std   = ("x2_std",   "mean"),   # approx — sufficient for EDA/features
        x2_count = ("x2_count", "sum"),
 
        t1_mean  = ("t1_mean",  "mean"),  t1_std  = ("t1_std",  "mean"),
        t2_mean  = ("t2_mean",  "mean"),  t2_std  = ("t2_std",  "mean"),
        t3_mean  = ("t3_mean",  "mean"),  t3_std  = ("t3_std",  "mean"),
        t4_mean  = ("t4_mean",  "mean"),  t4_std  = ("t4_std",  "mean"),
        t5_mean  = ("t5_mean",  "mean"),  t5_std  = ("t5_std",  "mean"),
        t6_mean  = ("t6_mean",  "mean"),  t6_std  = ("t6_std",  "mean"),
        t7_mean  = ("t7_mean",  "mean"),  t7_std  = ("t7_std",  "mean"),
        t8_mean  = ("t8_mean",  "mean"),  t8_std  = ("t8_std",  "mean"),
        t9_mean  = ("t9_mean",  "mean"),  t9_std  = ("t9_std",  "mean"),
        t10_mean = ("t10_mean", "mean"),  t10_std = ("t10_std", "mean"),
        t11_mean = ("t11_mean", "mean"),  t11_std = ("t11_std", "mean"),
        t12_mean = ("t12_mean", "mean"),  t12_std = ("t12_std", "mean"),
        t13_mean = ("t13_mean", "mean"),  t13_std = ("t13_std", "mean"),
 
        x1_mean    = ("x1_mean",    "mean"),
        x1_std     = ("x1_std",     "mean"),
        x3_mode    = ("x3_mode",    lambda s: s.mode().iloc[0] if len(s) else None),
        deviceType = ("deviceType", "first"),
    ).reset_index()

In [ ]:
OUT_PATH     = f"{DATA_FOLDER}/monthly_agg.parquet"
CHUNK_SIZE   = 500_000
t_start = time.time()
chunks  = []

for i, chunk in enumerate(
    pd.read_csv(DATA_PATH, dtype=DTYPES, chunksize=CHUNK_SIZE)
):
    t0   = time.time()
    agg  = aggregate_chunk(chunk)
    chunks.append(agg)
    elapsed = time.time() - t0
    total   = time.time() - t_start
    print(f"  chunk {i+1:>3}: {len(chunk):>8,} rows → "
        f"{len(agg):>5,} groups | "
        f"chunk: {elapsed:.1f}s | elapsed: {total/60:.1f}min")
    del chunk
    gc.collect()

print("\nCombining partial aggregations ...")
monthly = combine_partial_aggs(chunks)

In [ ]:
print("Merging device metadata ...")
devices = pd.read_csv(DEVICES_PATH)
monthly = monthly.merge(devices, on="deviceId", how="left")

In [29]:
monthly.to_parquet(OUT_PATH, index=False, engine="fastparquet")

print(f"\nDone in {(time.time()-t_start)/60:.1f} min")
print(f"  Shape   : {monthly.shape}")
print(f"  Columns : {list(monthly.columns)}")
print(f"  Saved   : {OUT_PATH}")
print(f"\nSanity check:")
print(monthly.groupby(["year","month"])["deviceId"].count().to_string())


Done in 7.5 min
  Shape   : (7701, 38)
  Columns : ['deviceId', 'year', 'month', 'x2_mean', 'x2_std', 'x2_count', 't1_mean', 't1_std', 't2_mean', 't2_std', 't3_mean', 't3_std', 't4_mean', 't4_std', 't5_mean', 't5_std', 't6_mean', 't6_std', 't7_mean', 't7_std', 't8_mean', 't8_std', 't9_mean', 't9_std', 't10_mean', 't10_std', 't11_mean', 't11_std', 't12_mean', 't12_std', 't13_mean', 't13_std', 'x1_mean', 'x1_std', 'x3_mode', 'deviceType', 'latitude', 'longitude']
  Saved   : /net/tscratch/people/tutorial237/EnsembleAi2026/task3/data/monthly_agg.parquet

Sanity check:
year  month
2024  10       596
      11       591
      12       590
2025  1        592
      2        588
      3        590
      4        595
      5        598
      6        598
      7        591
      8        592
      9        590
      10       590
